In [5]:
import numpy as np
from subprocess import PIPE, run
import matplotlib.pyplot as plt
import os
import textwrap
from waxx.control import ethernet_relay

class ExptBuilder():
    def __init__(self):
        self.__code_path__ = os.environ.get('code')
        self.__temp_exp_path__ = os.path.join(self.__code_path__, "k-exp", "kexp", "experiments", "ml_expt.py")

    def run_expt(self):
        expt_path = self.__temp_exp_path__
        run_expt_command = r"%kpy% & ar " + expt_path
        result = run(run_expt_command, stdout=PIPE, stderr=PIPE, universal_newlines=True, shell=True)
        print(result.returncode, result.stdout, result.stderr)
        os.remove(self.__temp_exp_path__)
        return result.returncode
    
    def write_experiment_to_file(self, program):
        with open(self.__temp_exp_path__, 'w') as file:
            file.write(program)
    
    def fringe_scan_expt(self, img_amp, freq_raman):
        script = textwrap.dedent(f"""
        from artiq.experiment import *
        from artiq.experiment import delay
        from kexp import Base, img_types, cameras
        import numpy as np
        from kexp.calibrations.tweezer import tweezer_vpd1_to_vpd2
        from kexp.calibrations.imaging import high_field_imaging_detuning
        from artiq.coredevice.sampler import Sampler
        from artiq.language import now_mu
        from kexp.util.artiq.async_print import aprint

        class hf_monitored_rabi(EnvExperiment, Base):

            def prepare(self):
                Base.__init__(self,setup_camera=True,
                            camera_select=cameras.andor,
                            save_data=True,
                            imaging_type=img_types.DISPERSIVE)

                self.p.v_pd_hf_tweezer_1064_rampdown2_end = .5
                self.p.frequency_raman_transition = {freq_raman}

                self.p.t_continuous_rabi = 450.e-6
                
                self.p.amp_imaging = {img_amp}

                self.p.hf_imaging_detuning = -568.e6 # 182.

                self.p.hf_imaging_detuning_mid = -514.e6 # -1 --> 0
                
                self.p.dimension_slm_mask = 20.e-6

                self.p.phase_slm_mask = 0.36 * np.pi

                self.p.t_tweezer_hold = 20.e-3

                self.p.t_tof = 20.e-6
                
                self.p.t_mot_load = 1.0
                
                self.p.N_repeats = 10

                self.scope = self.scope_data.add_siglent_scope("192.168.1.108", label='PD', arm=False)

                self.finish_prepare(shuffle=True)

            @kernel
            def scan_kernel(self):
                
                self.set_imaging_detuning(frequency_detuned = self.p.hf_imaging_detuning_mid)
                self.slm.write_phase_mask_kernel(phase=self.p.phase_slm_mask,dimension=self.p.dimension_slm_mask)
                self.imaging.set_power(self.p.amp_imaging)

                self.prepare_hf_tweezers(squeeze=False)

                self.raman.init(fraction_power = self.p.fraction_power_raman,
                                frequency_transition = self.p.frequency_raman_transition)

                self.ttl.raman_shutter.on()
                delay(10.e-3)
                self.ttl.line_trigger.wait_for_line_trigger()
                delay(4.7e-3)
                
                self.ttl.pd_scope_trig3.pulse(1.e-6)
                self.imaging.on()
                delay(3.e-6)

                self.raman.pulse(t=self.p.t_continuous_rabi)
                
                self.imaging.off()

                self.ttl.raman_shutter.off()
                
                self.set_imaging_detuning(frequency_detuned = self.p.hf_imaging_detuning)
                self.imaging.set_power(.2,reset_pid=True)

                delay(self.p.t_tweezer_hold)
                self.tweezer.off()

                delay(self.p.t_tof)

                self.abs_image()

                self.core.wait_until_mu(now_mu())
                self.scope.read_sweep(0)
                self.core.break_realtime()
                delay(30.e-3)

            @kernel
            def run(self):
                self.init_kernel()
                self.load_2D_mot(self.p.t_2D_mot_load_delay)
                self.scan()
                self.mot_observe()

            def analyze(self):
                import os
                expt_filepath = os.path.abspath(__file__)
                # aprint(self.scope._data)
                self.end(expt_filepath)
                """)
        return script

In [6]:
eBuilder = ExptBuilder()

In [7]:
img_amp = np.linspace(.2,2.5,100)

freq_raman = np.array([1.19499314e+08, 1.19503546e+08, 1.19507777e+08, 1.19512008e+08,
       1.19516239e+08, 1.19520471e+08, 1.19524702e+08, 1.19528933e+08,
       1.19533164e+08, 1.19537396e+08, 1.19541627e+08, 1.19545858e+08,
       1.19550090e+08, 1.19554321e+08, 1.19558552e+08, 1.19562783e+08,
       1.19567015e+08, 1.19571246e+08, 1.19575477e+08, 1.19579708e+08,
       1.19583940e+08, 1.19588171e+08, 1.19592402e+08, 1.19596633e+08,
       1.19600865e+08, 1.19605096e+08, 1.19609327e+08, 1.19613559e+08,
       1.19617790e+08, 1.19622021e+08, 1.19626252e+08, 1.19630484e+08,
       1.19634715e+08, 1.19638946e+08, 1.19643177e+08, 1.19647409e+08,
       1.19651640e+08, 1.19655871e+08, 1.19660103e+08, 1.19664334e+08,
       1.19668565e+08, 1.19672796e+08, 1.19677028e+08, 1.19681259e+08,
       1.19685490e+08, 1.19689721e+08, 1.19693953e+08, 1.19698184e+08,
       1.19702415e+08, 1.19706647e+08, 1.19710878e+08, 1.19715109e+08,
       1.19719340e+08, 1.19723572e+08, 1.19727803e+08, 1.19732034e+08,
       1.19736265e+08, 1.19740497e+08, 1.19744728e+08, 1.19748959e+08,
       1.19753190e+08, 1.19757422e+08, 1.19761653e+08, 1.19765884e+08,
       1.19770116e+08, 1.19774347e+08, 1.19778578e+08, 1.19782809e+08,
       1.19787041e+08, 1.19791272e+08, 1.19795503e+08, 1.19799734e+08,
       1.19803966e+08, 1.19808197e+08, 1.19812428e+08, 1.19816660e+08,
       1.19820891e+08, 1.19825122e+08, 1.19829353e+08, 1.19833585e+08,
       1.19837816e+08, 1.19842047e+08, 1.19846278e+08, 1.19850510e+08,
       1.19854741e+08, 1.19858972e+08, 1.19863203e+08, 1.19867435e+08,
       1.19871666e+08, 1.19875897e+08, 1.19880129e+08, 1.19884360e+08,
       1.19888591e+08, 1.19892822e+08, 1.19897054e+08, 1.19901285e+08,
       1.19905516e+08, 1.19909747e+08, 1.19913979e+08, 1.19918210e+08])


for f in range(len(img_amp)):
    print(img_amp[f],freq_raman[f])
    eBuilder.write_experiment_to_file(eBuilder.fringe_scan_expt(img_amp=img_amp[f],freq_raman = freq_raman[f]))
    eBuilder.run_expt()

0.2 119499314.0
0 Run id: 66663
 10 values of dummy. 10 total shots. 30 total images expected.
B:\_K\PotassiumData\2026-05-14\0066663_2026-05-14_06-48-18_hf_monitored_rabi.hdf5
Acknowledged camera ready signal.
Camera is ready.

Sent: {'mask': 'spot', 'center': [993, 820], 'phase': 0.36, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.36 pi, x-center = 993, y-center = 820

 Run ID: 66663

Sent: {'mask': 'spot', 'center': [993, 820], 'phase': 0.36, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.36 pi, x-center = 993, y-center = 820


Sent: {'mask': 'spot', 'center': [993, 820], 'phase': 0.36, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 20 um, phase = 0.36 pi, x-center = 993, y-center = 820


Sent: {'mask': 'spot', 'center': [993, 820], 'phase': 0.36, 'dimension': 20, 'initialize': False, 'spacing': 10, 'angle': 45}
